In [196]:
import pandas as pd
import numpy as np

# Function to generate lagged features
def generate_lagged_features(df, target, max_lag):
    df = df.copy()
    original_features = df.columns
    y = pd.DataFrame(df[target])
    for lag in range(1, max_lag + 1):
        df[f'{target}_lag_{lag}'] = df[target].shift(lag)
    lagged_df = df.drop(original_features, axis=1)
    lagged_df = pd.concat([lagged_df, y], axis=1)
    return lagged_df

# Function to select top lagged features based on maximum correlation
def select_top_lagged_features(lagged_df, target, top_n=10):
    correlation = lagged_df.corr()[target].sort_values(ascending=False)
    top_lagged_features = correlation.index[1:top_n+1]  # Exclude the target itself
    return top_lagged_features

def append_to_combined_df(combined_lag_train,combined_lag_test, lagged_train,lagged_test, target):
    combined_lag_train = pd.concat([combined_lag_train, lagged_train], axis=1)
    combined_lag_test = pd.concat([combined_lag_test, lagged_test], axis=1)
    return combined_lag_train,combined_lag_test
def append_to_combined_dict(results_dict,target):
    target = str(target)
    results_dict[target] = {'features':globals()[f'{target}_top_lagged_features']}
    return results_dict
# Paths for processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Define targets
targets = ['FEDFUNDS', 'GDP', 'CPIAUCSL', 'CUSR0000SAH1', 'CPILFESL', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'HOUST', 'DSPI', 
           'DGS2', 'DGS5', 'DGS10', 'AAA', 'BAA', 'WTISPLC', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI', 'PCEPI', 'PCEPILFE', 
           'PAYEMS', 'UNRATE', 'INDPRO', 'CUMFNS', 'USREC']

# Maximum lag to generate
max_lag = 10

original_features = train_combined.columns
# Initialize results list
results = []

lagged_df = pd.DataFrame()
combined_lag_train = pd.DataFrame()
combined_lag_test = pd.DataFrame()
lagged_features_dict = {}
results_dict = {}
# Loop through each target
for target in targets:
    # Generate lagged features
    lagged_train = generate_lagged_features(train_combined, target, max_lag)
    # Impute missing values using spline interpolation
    lagged_train = lagged_train.bfill()
    lagged_test = generate_lagged_features(test_combined, target, max_lag)
    lagged_test = lagged_test.bfill()

    # Find top lagged features based on correlation
    top_lagged_features = select_top_lagged_features(lagged_train, target)
    
    # Append lagged features to combined data
    combined_lag_train,combined_lag_test = append_to_combined_df(combined_lag_train,combined_lag_test, lagged_train, lagged_test, target)
    # Store the top lagged features in a variable named after the target
    globals()[f'{target}_top_lagged_features'] = top_lagged_features.tolist()

    # Collect results for display
    results.append({
        'target': target,
        'top_lagged_features': globals()[f'{target}_top_lagged_features']
    })

    results_dict = append_to_combined_dict(results_dict, target)


# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results)

# Print the DataFrame
print("\nTop lagged features for each target:")




Top lagged features for each target:


In [193]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from best_params import xgboost_params, lightgbm_params

# Function to compute MSE scores without feature scaling
def compute_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf
    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    mse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(verbosity=-1, **lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse_scores[model_name] = mean_squared_error(y_test, y_pred)

    aggregated_mse = sum(mse_scores.values())

    return mse_scores, aggregated_mse

# Paths for processed data
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)

lagged_features = pd.DataFrame(results_dict).T.to_dict()['features']

features_to_add = combined_lag_train.columns
# Loop through each target
results = []

for target, features in lagged_features.items():
    print(f"\nProcessing target: {target}")
    x_features = [col for col in train_combined.columns if col in train_combined.columns]
    # remove target from features
    x_features.remove(target)
    X_train = train_combined[x_features].copy()
    X_test = test_combined[x_features].copy()
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]

    # Compute initial baseline MSE
    valid_features = [col for col in X_train.columns if col in X_train.columns]
    baseline_mse_scores, aggregated_baseline_mse = compute_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    variance = np.var(y_test)
    baseline_mse_var = aggregated_baseline_mse / variance
    # Calculate Baseline MSE
    print(f"\nBaseline MSE for target: {target}")
    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")

    print(f"Initial aggregated baseline MSE: {aggregated_baseline_mse}")
    print(f"Initial MSE/Variance: {baseline_mse_var}")
    print(f"Initial MSE for XGBoost: {baseline_mse_scores['XGBoost']}")
    print(f"Initial MSE for LightGBM: {baseline_mse_scores['LightGBM']}")
    # Access the features for the specific target key in the dictionary
    features_to_add = lagged_features[target]

    # Evaluate adding lagged features
    for feature in features_to_add:
            X_train_plus = X_train.copy()
            X_test_plus = X_test.copy()
            X_train_plus[feature] = combined_lag_train[feature]
            X_test_plus[feature] = combined_lag_test[feature]
                # Update valid_features with the new feature
            valid_features.append(feature)
            print(f"\nAdding feature: {feature}")

            # Compute MSE after adding the feature
            mse_scores, aggregated_mse = compute_mse_scores(X_train_plus, X_test_plus, y_train, y_test, valid_features)
            improvement = aggregated_baseline_mse - aggregated_mse
            pct_improvement = (improvement / aggregated_baseline_mse) * 100
            improvement_var = improvement / variance

            print(f"New aggregated MSE after adding {feature}: {aggregated_mse}")
            print(f"% Improvement: {pct_improvement}")
            print(f"% Improvement in MSE/Variance: {(improvement_var / baseline_mse_var) * 100}")
            print(f"New MSE for XGBoost: {mse_scores['XGBoost']}")
            print(f"New MSE for LightGBM: {mse_scores['LightGBM']}")

            X_train_plus = X_train.copy()
            X_test_plus = X_test.copy()
            valid_features.remove(feature)

    results.append({
        'target': target,
        'initial_mse_xgboost': baseline_mse_scores['XGBoost'],
        'initial_mse_lightgbm': baseline_mse_scores['LightGBM'],
        'final_mse_xgboost': mse_scores['XGBoost'],
        'final_mse_lightgbm': mse_scores['LightGBM'],
        'initial_aggregated_mse': aggregated_baseline_mse,
        'final_aggregated_mse': aggregated_mse,
        'improvement': aggregated_baseline_mse - aggregated_mse,
        'final_feature_space': valid_features + features_to_add
    })

print("\nFeature addition completed for all targets.")



Processing target: FEDFUNDS

Baseline MSE for target: FEDFUNDS
Initial aggregated baseline MSE: 0.018823862107365842
Initial aggregated baseline MSE: 0.018823862107365842
Initial MSE/Variance: FEDFUNDS    2.495556
dtype: float64
Initial MSE for XGBoost: 0.008660657426180808
Initial MSE for LightGBM: 0.010163204681185034

Adding feature: FEDFUNDS_lag_1
New aggregated MSE after adding FEDFUNDS_lag_1: 0.02151964327788721
% Improvement: -14.321084351050898
% Improvement in MSE/Variance: FEDFUNDS   -14.321084
dtype: float64
New MSE for XGBoost: 0.010498688009954479
New MSE for LightGBM: 0.011020955267932734

Adding feature: FEDFUNDS_lag_5
New aggregated MSE after adding FEDFUNDS_lag_5: 0.019493799466421793
% Improvement: -3.5589793169691872
% Improvement in MSE/Variance: FEDFUNDS   -3.558979
dtype: float64
New MSE for XGBoost: 0.009428079197240586
New MSE for LightGBM: 0.010065720269181207

Adding feature: FEDFUNDS_lag_2
New aggregated MSE after adding FEDFUNDS_lag_2: 0.021365818473430553


In [209]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from best_params import xgboost_params, lightgbm_params

# Function to compute RMSE and MSE scores without feature scaling
def compute_rmse_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, np.inf
    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    rmse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(verbosity=-1, **lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mse_scores[model_name] = mse
        rmse_scores[model_name] = rmse

    aggregated_mse = sum(mse_scores.values())
    aggregated_rmse = sum(rmse_scores.values())

    return mse_scores, rmse_scores, aggregated_mse, aggregated_rmse

# Paths for processed data
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)

lagged_features = pd.DataFrame(results_dict).T.to_dict()['features']

# Initialize results list
all_targets_results = []

# Loop through each target
for target, features in lagged_features.items():
    print(f"\nProcessing target: {target}")
    x_features = [col for col in train_combined.columns if col in train_combined.columns]
    # remove target from features
    x_features.remove(target)
    X_train = train_combined[x_features].copy()
    X_test = test_combined[x_features].copy()
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]

    # Compute initial baseline RMSE and MSE
    valid_features = [col for col in X_train.columns if col in X_train.columns]
    baseline_mse_scores, baseline_rmse_scores, aggregated_baseline_mse, aggregated_baseline_rmse = compute_rmse_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    variance = np.var(y_test)
    baseline_rmse_var = aggregated_baseline_rmse / variance
    
    print(f"\nBaseline RMSE for target: {target}")
    print(f"Initial aggregated baseline RMSE: {aggregated_baseline_rmse}")
    
    feature_performance = []
    
    # Evaluate adding lagged features
    for feature in features:
        X_train_plus = X_train.copy()
        X_test_plus = X_test.copy()
        X_train_plus[feature] = combined_lag_train[feature]
        X_test_plus[feature] = combined_lag_test[feature]
        valid_features.append(feature)

        # Compute RMSE and MSE after adding the feature
        mse_scores, rmse_scores, aggregated_mse, aggregated_rmse = compute_rmse_mse_scores(X_train_plus, X_test_plus, y_train, y_test, valid_features)
        rmse_improvement = aggregated_baseline_rmse - aggregated_rmse
        pct_improvement_rmse = (rmse_improvement / aggregated_baseline_rmse) * 100
        rmse_var_improvement = (rmse_improvement / variance)

        # Store the feature and its performance
        feature_performance.append({
            'feature': feature,
            'rmse_improvement': rmse_improvement,
            'pct_improvement_rmse': pct_improvement_rmse,
            'rmse_var_improvement': (rmse_var_improvement / baseline_rmse_var) * 100
        })

        print(f"Feature: {feature}, % Improvement in RMSE: {pct_improvement_rmse}")
        
        # Revert valid_features list to avoid adding the feature multiple times
        valid_features.remove(feature)
    
    # Sort by RMSE improvement and select the top three features
    top_features = sorted(feature_performance, key=lambda x: x['pct_improvement_rmse'], reverse=True)[:3]
    
    # Add results to all_targets_results list
    for top_feature in top_features:
        all_targets_results.append({
            'target': target,
            'feature': top_feature['feature'],
            'rmse_improvement': top_feature['rmse_improvement'],
            'pct_improvement_rmse': top_feature['pct_improvement_rmse'],
            'rmse_var_improvement': top_feature['rmse_var_improvement']
        })

# Convert results to DataFrame and save to CSV
results_df = pd.DataFrame(all_targets_results)
results_df.to_csv('top_features_rmse_results.csv', index=False)

print("Feature addition completed and top features saved.")



Processing target: FEDFUNDS

Baseline RMSE for target: FEDFUNDS
Initial aggregated baseline RMSE: 0.19387537406004413
Feature: FEDFUNDS_lag_1, % Improvement in RMSE: -6.998552143967937
Feature: FEDFUNDS_lag_5, % Improvement in RMSE: -1.8315816271108323
Feature: FEDFUNDS_lag_2, % Improvement in RMSE: -6.615123356373827
Feature: FEDFUNDS_lag_7, % Improvement in RMSE: 2.6610579523682403
Feature: FEDFUNDS_lag_4, % Improvement in RMSE: 0.36686750284783537
Feature: FEDFUNDS_lag_6, % Improvement in RMSE: 4.777277509825918
Feature: FEDFUNDS_lag_10, % Improvement in RMSE: 0.5829135173550191
Feature: FEDFUNDS_lag_3, % Improvement in RMSE: 6.28167460404078
Feature: FEDFUNDS_lag_9, % Improvement in RMSE: 4.39622686836939
Feature: FEDFUNDS_lag_8, % Improvement in RMSE: 1.729738447890421

Processing target: GDP

Baseline RMSE for target: GDP
Initial aggregated baseline RMSE: 0.0338789404561649
Feature: GDP_lag_1, % Improvement in RMSE: 29.79455597849781
Feature: GDP_lag_2, % Improvement in RMSE: 2.

In [220]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from best_params import xgboost_params, lightgbm_params

# Function to compute RMSE and MSE scores without feature scaling
def compute_rmse_mse_scores(X_train, X_test, y_train, y_test, features):
    if not features:
        return {'XGBoost': np.inf, 'LightGBM': np.inf}, np.inf, np.inf
    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna().values.ravel()
    y_test = y_test.dropna().values.ravel()

    mse_scores = {'XGBoost': [], 'LightGBM': []}
    rmse_scores = {'XGBoost': [], 'LightGBM': []}

    models = {
        'XGBoost': XGBRegressor(**xgboost_params),
        'LightGBM': LGBMRegressor(verbosity=-1, **lightgbm_params)
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mse_scores[model_name] = mse
        rmse_scores[model_name] = rmse

    aggregated_mse = sum(mse_scores.values())
    aggregated_rmse = sum(rmse_scores.values())

    return mse_scores, rmse_scores, aggregated_mse, aggregated_rmse

# Paths for processed data
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)

lagged_features = pd.DataFrame(results_dict).T.to_dict()['features']

# Initialize results list
all_targets_results = []

# Loop through each target
for target, features in lagged_features.items():
    print(f"\nProcessing target: {target}")
    x_features = [col for col in train_combined.columns if col in train_combined.columns]
    # remove target from features
    x_features.remove(target)
    X_train = train_combined[x_features].copy()
    X_test = test_combined[x_features].copy()
    y_train = train_combined[[target]]
    y_test = test_combined[[target]]

    # Compute initial baseline RMSE and MSE
    valid_features = [col for col in X_train.columns if col in X_train.columns]
    baseline_mse_scores, baseline_rmse_scores, aggregated_baseline_mse, aggregated_baseline_rmse = compute_rmse_mse_scores(X_train, X_test, y_train, y_test, valid_features)
    variance = np.var(y_test)
    baseline_rmse_var = aggregated_baseline_rmse / variance
    baseline_mse_var = aggregated_baseline_mse / variance

    print(f"\nBaseline RMSE for target: {target}")
    print(f"Initial aggregated baseline RMSE: {aggregated_baseline_rmse}")
    
    feature_performance = []
    
    # Evaluate adding lagged features
    for feature in features:
        X_train_plus = X_train.copy()
        X_test_plus = X_test.copy()
        X_train_plus[feature] = combined_lag_train[feature]
        X_test_plus[feature] = combined_lag_test[feature]
        valid_features.append(feature)

        # Compute RMSE and MSE after adding the feature
        mse_scores, rmse_scores, aggregated_mse, aggregated_rmse = compute_rmse_mse_scores(X_train_plus, X_test_plus, y_train, y_test, valid_features)
        mse_improvement = aggregated_baseline_mse - aggregated_mse
        rmse_improvement = aggregated_baseline_rmse - aggregated_rmse
        pct_improvement_mse = (mse_improvement / aggregated_baseline_mse) * 100
        pct_improvement_rmse = (rmse_improvement / aggregated_baseline_rmse) * 100

        # Combine MSE and RMSE improvements using a weighted average or other method
        combined_score = (pct_improvement_mse + pct_improvement_rmse) / 2
        
        # Store the feature and its performance
        feature_performance.append({
            'feature': feature,
            'mse_improvement': mse_improvement,
            'rmse_improvement': rmse_improvement,
            'pct_improvement_mse': pct_improvement_mse,
            'pct_improvement_rmse': pct_improvement_rmse,
            'combined_score': combined_score
        })

        print(f"Feature: {feature}, Combined Score (MSE & RMSE): {combined_score}")
        
        # Revert valid_features list to avoid adding the feature multiple times
        valid_features.remove(feature)
    
    # Sort by combined score and select the top three features
    top_features = sorted(feature_performance, key=lambda x: x['combined_score'], reverse=True)[:3]
    
    # Add results to all_targets_results list
    for top_feature in top_features:
        all_targets_results.append({
            'target': target,
            'feature': top_feature['feature'],
            'mse_improvement': top_feature['mse_improvement'],
            'rmse_improvement': top_feature['rmse_improvement'],
            'pct_improvement_mse': top_feature['pct_improvement_mse'],
            'pct_improvement_rmse': top_feature['pct_improvement_rmse'],
            'combined_score': top_feature['combined_score']
        })

# Convert results to DataFrame and save to CSV
results_df = pd.DataFrame(all_targets_results)
results_df.to_csv('top_features_combined_rmse_mse_results.csv', index=False)

print("Feature addition completed and top features saved.")



Processing target: FEDFUNDS

Baseline RMSE for target: FEDFUNDS
Initial aggregated baseline RMSE: 0.19387537406004413
Feature: FEDFUNDS_lag_1, Combined Score (MSE & RMSE): -10.659818247509417
Feature: FEDFUNDS_lag_5, Combined Score (MSE & RMSE): -2.6952804720400096
Feature: FEDFUNDS_lag_2, Combined Score (MSE & RMSE): -10.059513938477949
Feature: FEDFUNDS_lag_7, Combined Score (MSE & RMSE): 3.9462216092736906
Feature: FEDFUNDS_lag_4, Combined Score (MSE & RMSE): 0.6210974486129375
Feature: FEDFUNDS_lag_6, Combined Score (MSE & RMSE): 7.116946006856335
Feature: FEDFUNDS_lag_10, Combined Score (MSE & RMSE): 0.8760707136595677
Feature: FEDFUNDS_lag_3, Combined Score (MSE & RMSE): 9.26859056673888
Feature: FEDFUNDS_lag_9, Combined Score (MSE & RMSE): 6.541677420885451
Feature: FEDFUNDS_lag_8, Combined Score (MSE & RMSE): 2.6468590770170373

Processing target: GDP

Baseline RMSE for target: GDP
Initial aggregated baseline RMSE: 0.0338789404561649
Feature: GDP_lag_1, Combined Score (MSE & R

KeyboardInterrupt: 

In [221]:
display(combined_lag_train)
combined_lag_train.to_csv('data/engineered/combined_lag_train.csv')
combined_lag_test.to_csv('data/engineered/combined_lag_test.csv')

,FEDFUNDS_lag_1,FEDFUNDS_lag_2,FEDFUNDS_lag_3,FEDFUNDS_lag_4,FEDFUNDS_lag_5,FEDFUNDS_lag_6,FEDFUNDS_lag_7,FEDFUNDS_lag_8,FEDFUNDS_lag_9,FEDFUNDS_lag_10,FEDFUNDS,GDP_lag_1,GDP_lag_2,GDP_lag_3,GDP_lag_4,GDP_lag_5,GDP_lag_6,GDP_lag_7,GDP_lag_8,GDP_lag_9,GDP_lag_10,GDP,CPIAUCSL_lag_1,CPIAUCSL_lag_2,CPIAUCSL_lag_3,CPIAUCSL_lag_4,CPIAUCSL_lag_5,CPIAUCSL_lag_6,CPIAUCSL_lag_7,CPIAUCSL_lag_8,CPIAUCSL_lag_9,CPIAUCSL_lag_10,CPIAUCSL,CUSR0000SAH1_lag_1,CUSR0000SAH1_lag_2,CUSR0000SAH1_lag_3,CUSR0000SAH1_lag_4,CUSR0000SAH1_lag_5,CUSR0000SAH1_lag_6,CUSR0000SAH1_lag_7,CUSR0000SAH1_lag_8,CUSR0000SAH1_lag_9,CUSR0000SAH1_lag_10,CUSR0000SAH1,CPILFESL_lag_1,CPILFESL_lag_2,CPILFESL_lag_3,CPILFESL_lag_4,CPILFESL_lag_5,CPILFESL_lag_6,CPILFESL_lag_7,CPILFESL_lag_8,CPILFESL_lag_9,CPILFESL_lag_10,CPILFESL,PCE_lag_1,PCE_lag_2,PCE_lag_3,PCE_lag_4,PCE_lag_5,PCE_lag_6,PCE_lag_7,PCE_lag_8,PCE_lag_9,PCE_lag_10,PCE,PRFI_lag_1,PRFI_lag_2,PRFI_lag_3,PRFI_lag_4,PRFI_lag_5,PRFI_lag_6,PRFI_lag_7,PRFI_lag_8,PRFI_lag_9,PRFI_lag_10,PRFI,PNFI_lag_1,PNFI_lag_2,PNFI_lag_3,PNFI_lag_4,PNFI_lag_5,PNFI_lag_6,PNFI_lag_7,PNFI_lag_8,PNFI_lag_9,PNFI_lag_10,PNFI,EXPGS_lag_1,EXPGS_lag_2,EXPGS_lag_3,EXPGS_lag_4,EXPGS_lag_5,EXPGS_lag_6,EXPGS_lag_7,EXPGS_lag_8,EXPGS_lag_9,EXPGS_lag_10,EXPGS,HOUST_lag_1,HOUST_lag_2,HOUST_lag_3,HOUST_lag_4,HOUST_lag_5,HOUST_lag_6,HOUST_lag_7,HOUST_lag_8,HOUST_lag_9,HOUST_lag_10,HOUST,DSPI_lag_1,DSPI_lag_2,DSPI_lag_3,DSPI_lag_4,DSPI_lag_5,DSPI_lag_6,DSPI_lag_7,DSPI_lag_8,DSPI_lag_9,DSPI_lag_10,DSPI,DGS2_lag_1,DGS2_lag_2,DGS2_lag_3,DGS2_lag_4,DGS2_lag_5,DGS2_lag_6,DGS2_lag_7,DGS2_lag_8,DGS2_lag_9,DGS2_lag_10,DGS2,DGS5_lag_1,DGS5_lag_2,DGS5_lag_3,DGS5_lag_4,DGS5_lag_5,DGS5_lag_6,DGS5_lag_7,DGS5_lag_8,DGS5_lag_9,DGS5_lag_10,DGS5,DGS10_lag_1,DGS10_lag_2,DGS10_lag_3,DGS10_lag_4,DGS10_lag_5,DGS10_lag_6,DGS10_lag_7,DGS10_lag_8,DGS10_lag_9,DGS10_lag_10,DGS10,AAA_lag_1,AAA_lag_2,AAA_lag_3,AAA_lag_4,AAA_lag_5,AAA_lag_6,AAA_lag_7,AAA_lag_8,AAA_lag_9,AAA_lag_10,AAA,BAA_lag_1,BAA_lag_2,BAA_lag_3,BAA_lag_4,BAA_lag_5,BAA_lag_6,BAA_lag_7,BAA_lag_8,BAA_lag_9,BAA_lag_10,BAA,WTISPLC_lag_1,WTISPLC_lag_2,WTISPLC_lag_3,WTISPLC_lag_4,WTISPLC_lag_5,WTISPLC_lag_6,WTISPLC_lag_7,WTISPLC_lag_8,WTISPLC_lag_9,WTISPLC_lag_10,WTISPLC,IMPGS_lag_1,IMPGS_lag_2,IMPGS_lag_3,IMPGS_lag_4,IMPGS_lag_5,IMPGS_lag_6,IMPGS_lag_7,IMPGS_lag_8,IMPGS_lag_9,IMPGS_lag_10,IMPGS,GCE_lag_1,GCE_lag_2,GCE_lag_3,GCE_lag_4,GCE_lag_5,GCE_lag_6,GCE_lag_7,GCE_lag_8,GCE_lag_9,GCE_lag_10,GCE,FGCE_lag_1,FGCE_lag_2,FGCE_lag_3,FGCE_lag_4,FGCE_lag_5,FGCE_lag_6,FGCE_lag_7,FGCE_lag_8,FGCE_lag_9,FGCE_lag_10,FGCE,GDPCTPI_lag_1,GDPCTPI_lag_2,GDPCTPI_lag_3,GDPCTPI_lag_4,GDPCTPI_lag_5,GDPCTPI_lag_6,GDPCTPI_lag_7,GDPCTPI_lag_8,GDPCTPI_lag_9,GDPCTPI_lag_10,GDPCTPI,PCEPI_lag_1,PCEPI_lag_2,PCEPI_lag_3,PCEPI_lag_4,PCEPI_lag_5,PCEPI_lag_6,PCEPI_lag_7,PCEPI_lag_8,PCEPI_lag_9,PCEPI_lag_10,PCEPI,PCEPILFE_lag_1,PCEPILFE_lag_2,PCEPILFE_lag_3,PCEPILFE_lag_4,PCEPILFE_lag_5,PCEPILFE_lag_6,PCEPILFE_lag_7,PCEPILFE_lag_8,PCEPILFE_lag_9,PCEPILFE_lag_10,PCEPILFE,PAYEMS_lag_1,PAYEMS_lag_2,PAYEMS_lag_3,PAYEMS_lag_4,PAYEMS_lag_5,PAYEMS_lag_6,PAYEMS_lag_7,PAYEMS_lag_8,PAYEMS_lag_9,PAYEMS_lag_10,PAYEMS,UNRATE_lag_1,UNRATE_lag_2,UNRATE_lag_3,UNRATE_lag_4,UNRATE_lag_5,UNRATE_lag_6,UNRATE_lag_7,UNRATE_lag_8,UNRATE_lag_9,UNRATE_lag_10,UNRATE,INDPRO_lag_1,INDPRO_lag_2,INDPRO_lag_3,INDPRO_lag_4,INDPRO_lag_5,INDPRO_lag_6,INDPRO_lag_7,INDPRO_lag_8,INDPRO_lag_9,INDPRO_lag_10,INDPRO,CUMFNS_lag_1,CUMFNS_lag_2,CUMFNS_lag_3,CUMFNS_lag_4,CUMFNS_lag_5,CUMFNS_lag_6,CUMFNS_lag_7,CUMFNS_lag_8,CUMFNS_lag_9,CUMFNS_lag_10,CUMFNS,USREC_lag_1,USREC_lag_2,USREC_lag_3,USREC_lag_4,USREC_lag_5,USREC_lag_6,USREC_lag_7,USREC_lag_8,USREC_lag_9,USREC_lag_10,USREC
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1976-02-0

In [218]:
results_df
results_df.to_csv('top_lagged_feature_results.csv', index=False)
csv_output_df =pd.DataFrame()
csv_output_df['target'] = results_df['target']
csv_output_df['feature'] = results_df['feature']
# display(csv_output_df)
# csv_output_df.to_csv('top_lagged_feature_results.csv', index=False)

In [219]:
results_df

# Selecting the best features for each target
targets = results_df['target'].unique()
for target in targets:
    target_df = results_df[results_df['target'] == target]
    best_mse_feature = target_df.loc[target_df['mse_improvement'].idxmax()]
    print(f"\nBest MSE feature for target {target}: {best_mse_feature['feature']}")
    print(f"MSE Improvement: {best_mse_feature['mse_improvement']}")
    print(f"% Improvement: {best_mse_feature['pct_improvement']}")
    print(f"RMSE Improvement: {best_mse_feature['rmse_improvement']}")
    best_rmse_feature = target_df.loc[target_df['rmse_improvement'].idxmax()]
    print(f"\nBest RMSE feature for target {target}: {best_rmse_feature['feature']}")
    print(f"MSE Improvement: {best_rmse_feature['mse_improvement']}")
    print(f"% Improvement: {best_rmse_feature['pct_improvement']}")
    print(f"RMSE Improvement: {best_rmse_feature['rmse_improvement']}")
    best_mse_var_feature = target_df.loc[target_df['mse_var_improvement'].idxmax()]
    print(f"\nBest MSE/Variance feature for target {target}: {best_mse_var_feature['feature']}")
    print(f"MSE Improvement: {best_mse_var_feature['mse_improvement']}")
    print(f"% Improvement: {best_mse_var_feature['pct_improvement']}")
    


Best MSE feature for target FEDFUNDS: FEDFUNDS_lag_3
MSE Improvement: 0.002306959649660434


KeyError: 'pct_improvement'

In [84]:
lagged_features_dict ={}
for target, features_to_add in results_df.items():
    # print(target)
    # print(features_to_add)
    # Create a dictionary, where the keys are the target names and the values are the top lagged features
    lagged_features_dict[target] = features_to_add
    # display dict 
    display(lagged_features_dict)

{'target': 0         FEDFUNDS
 1              GDP
 2         CPIAUCSL
 3     CUSR0000SAH1
 4         CPILFESL
 5              PCE
 6             PRFI
 7             PNFI
 8            EXPGS
 9            HOUST
 10            DSPI
 11            DGS2
 12            DGS5
 13           DGS10
 14             AAA
 15             BAA
 16         WTISPLC
 17           IMPGS
 18             GCE
 19            FGCE
 20         GDPCTPI
 21           PCEPI
 22        PCEPILFE
 23          PAYEMS
 24          UNRATE
 25          INDPRO
 26          CUMFNS
 27           USREC
 Name: target, dtype: object}

{'target': 0         FEDFUNDS
 1              GDP
 2         CPIAUCSL
 3     CUSR0000SAH1
 4         CPILFESL
 5              PCE
 6             PRFI
 7             PNFI
 8            EXPGS
 9            HOUST
 10            DSPI
 11            DGS2
 12            DGS5
 13           DGS10
 14             AAA
 15             BAA
 16         WTISPLC
 17           IMPGS
 18             GCE
 19            FGCE
 20         GDPCTPI
 21           PCEPI
 22        PCEPILFE
 23          PAYEMS
 24          UNRATE
 25          INDPRO
 26          CUMFNS
 27           USREC
 Name: target, dtype: object,
 'top_lagged_features': 0                         [FEDFUNDS_lag_1, FEDFUNDS_lag_5, FEDFUNDS_lag_2, FEDFUNDS_lag_7, FEDFUNDS_lag_4]
 1                                                 [GDP_lag_1, GDP_lag_2, GDP_lag_8, GDP_lag_9, GDP_lag_10]
 2                         [CPIAUCSL_lag_1, CPIAUCSL_lag_2, CPIAUCSL_lag_3, CPIAUCSL_lag_4, CPIAUCSL_lag_5]
 3     [CUSR0000SAH1_lag_1, CUSR0000SAH1_lag_2, CUSR0